In [1]:
# regress out confounds 
# import some libraries first
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import statsmodels.api as sm
import statsmodels.api
from statsmodels.formula.api import ols
from sklearn.linear_model import LinearRegression


In [2]:
def load_csv_file(filename):
    df = pd.read_csv(filename)
    return df

In [3]:
file_path = '/add/your/path/here'

In [4]:
aka_file = file_path + '/aka_all_data_word2vecL2Norm_data.csv'
madan_file = file_path + '/madan_words_L2norm.csv'
cox_file = file_path + '/coxetal_words_L2norm.csv'
subtlex_file = file_path + '/SUBTLEX_US_frequency_list.csv'
dymarska_file = file_path + '/dymarska_memorability_L2norm_dav.csv'

cox = False
aka = True
dymarska = False

In [5]:
if cox: 
    cox_data = load_csv_file(cox_file)
    data = cox_data
elif aka:
    aka_data = load_csv_file(aka_file)
    data = aka_data
elif dymarska:
    dymarska_data = load_csv_file(dymarska_file)
    dymarska_data['word'] = dymarska_data['word'].apply(lambda x: x.lower() if isinstance(x, str) else x)
    data = dymarska_data
    
else:
    madan_data = load_csv_file(madan_file)
    data = madan_data

subtlex_data = load_csv_file(subtlex_file)

In [8]:
# add new column to data dataframe with subtlex_freq values per word 
data['subtlex_freq'] = None

#reate a dictionary from the 'subtlex_data' for faster lookup
subtlex_dict = dict(zip(subtlex_data['Word'], subtlex_data['Lg10WF']))

for index, row in data.iterrows():
    word = row['word']
    if word in subtlex_dict:
        data.at[index, 'subtlex_freq'] = subtlex_dict[word]
    else:
        data.at[index, 'subtlex_freq'] = None  



In [9]:
#ensure columns are numeric
data['subtlex_freq'] = pd.to_numeric(data['subtlex_freq'], errors='coerce')

In [11]:
#X_freq = sm.add_constant(data['subtlex_freq'])  #adding constant term for the regression
X_freq = sm.add_constant(data['Valence'])  #adding constant term for the regression
#X_freq = sm.add_constant(data['Size'])  # adding constant term for the regression

model_L2norm = sm.OLS(data['vector_length_word2vec'], X_freq).fit()  
#  store the residuals
data['L2norm_residual'] = model_L2norm.resid  

In [1]:
if cox: 
    model_final = sm.OLS(data['recognition_memorability'], sm.add_constant(data['L2norm_residual'])).fit()
elif aka: 
    model_final = sm.OLS(data['actual_recognition'], sm.add_constant(data['L2norm_residual'])).fit()
elif dymarska: 
    model_final = sm.OLS(data['both_hit'], sm.add_constant(data['L2norm_residual'])).fit()
else: 
    model_final = sm.OLS(data['pRecall'], sm.add_constant(data['L2norm_residual'])).fit()


print(model_final.summary())  

In [2]:
# plot the results
sns.lmplot(x='actual_recognition', y='L2norm_residual', data=data, ci=None, 
           scatter_kws={'color': 'brown'}, line_kws={'color': 'black'})


# Display the plot
plt.xlabel('Recognition Memorability')
plt.ylabel('L2 Norm Residual')
plt.title(' Aka data')
plt.show()

In [ ]:
correlation_coefficient, p_value = scipy.stats.spearmanr(cox_data['recall_memorability'], cox_data['recognition_memorability'])
print("Pearson correlation coefficient:", correlation_coefficient)
print("P-value recall:", p_value)

## correlation matrices

In [4]:
# correlation matrix 
correlation_matrix = aka_data[['actual_recognition', 'vector_length_word2vec', 'WFlog', 'Animacy', 'Size', 'Valence', 'Arousal', 'nLet', 'nSyl', 'Concreteness', 'BOI', 'NoF']].corr(method='spearman')
p_values_matrix = aka_data[['actual_recognition', 'vector_length_word2vec', 'WFlog', 'Animacy', 'Size', 'Valence', 'Arousal', 'nLet', 'nSyl', 'Concreteness', 'BOI', 'NoF']].corr(method=lambda x, y: scipy.stats.spearmanr(x, y)[1])
print(correlation_matrix)
print(p_values_matrix)

In [3]:
# heatmap plot of correlations and p-values
plt.figure(figsize=(10, 7))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, 
            square=True, mask=False, cbar=True, center=0, vmin=-1, vmax=1,
            xticklabels=correlation_matrix.columns, yticklabels=correlation_matrix.columns)
plt.title('Correlation Matrix')
plt.show()

plt.figure(figsize=(10, 7))
sns.heatmap(p_values_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('P-values of Correlation Matrix')
plt.show()